In [15]:
%run ../utility/common_functions.ipynb

In [16]:
session = get_active_session()
session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
session.sql("USE SCHEMA SILVER").collect()

run_start_time = datetime.now()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

pipeline_name = "Customer Billing Pipeline"
task_name = "Bronze To Silver Layer Transformation"

In [17]:
# Read all Silver YAML files
# 1. SESSION AND SILVER YAML FILE DISCOVERY
session = get_active_session()

def C(column_name):
    """Reference an uppercase Snowflake column name."""
    escaped_name = column_name.strip('"').upper().replace('"', '""')
    return col(f'"{escaped_name}"')

session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
session.sql("USE SCHEMA SILVER").collect()

silver_yaml_path = ("@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/config/yaml/silver/")
all_files = session.sql(f"LIST {silver_yaml_path}").collect()
all_files = [
    f for f in all_files
    if f["name"].lower().endswith((".yaml", ".yml"))
]

if not all_files:
    raise FileNotFoundError(f"No Silver YAML files found in {silver_yaml_path}")
print(f"Silver YAML files discovered: {len(all_files)}")

for f in all_files:
    print(f["name"].rsplit("/", 1)[-1])

# 2. PROCESS EACH SILVER YAML CONFIGURATION
for yaml_file in all_files:
    inserted_count = 0
    updated_count = 0
    duplicate_count = 0

    yaml_file_name = yaml_file["name"].rsplit("/", 1)[-1]
    table_name = yaml_file_name.rsplit(".", 1)[0]
    try:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

        # Build Snowflake stage path
        yaml_stage_file = (silver_yaml_path.rstrip("/") + "/" + yaml_file_name)

        # Read YAML as text and parse it
        yaml_text = open_file_read_data(yaml_stage_file)
        config = yaml.safe_load(yaml_text)["table"]

        # 3. READ TABLE CONFIGURATION
        table_name = config["name"]
        source_schema = config["source_schema_name"]
        source_table_name = config.get("source_table_name", table_name)
        target_schema = config["schema_name"]
        source_table = (f"CUSTOMER_BILLING_DB.{source_schema}.{source_table_name}")
        target_table = (f"CUSTOMER_BILLING_DB.{target_schema}.{table_name}")
        columns = config["columns"]
        column_names = [c["name"] for c in columns]
        column_types = {
            c["name"]: c["data_type"].upper()
            for c in columns
        }
        business_keys = config["business_key"]
        hash_config = config["hash_key"]
        hash_name = hash_config.get("name", "hash_key")
        excluded_columns = config["excluded_columns"]
        scd = config["scd"]
        inserted_col = scd["inserted_date_column"]
        updated_col = scd["updated_date_column"]
        latest_col = scd["latest_flag_column"]
        latest_value = scd.get("latest_flag_value", 1)

        # 4. IDENTIFY DATA AND METADATA COLUMNS
        metadata_columns = [hash_name,inserted_col,updated_col,latest_col]
        data_columns = [
            c for c in column_names
            if c not in metadata_columns
        ]
        hash_columns = [
            c for c in column_names
            if c not in excluded_columns
        ]

        if not business_keys:
            raise ValueError(f"No business keys configured for {table_name}")

        if not hash_columns:
            raise ValueError(f"No hash columns configured for {table_name}")

        print("\n" + "=" * 60)
        print(f"Processing Silver table: {target_table}")
        print(f"Source table: {source_table}")
        print(f"Business keys: {business_keys}")
        print(f"Hash columns: {hash_columns}")
        print("=" * 60)
        process_log("Start",table_name=table_name,layer="silver")

        # 5. READ BRONZE DATA AND APPLY YAML DATA TYPES
        # 5. READ BRONZE DATA AND APPLY YAML DATA TYPES
        bronze = session.table(source_table).select(*[C(c).alias(c.strip('"').upper()) for c in data_columns])
        for c in data_columns:
            upper_col = c.strip('"').upper()
            bronze = bronze.with_column(upper_col,C(upper_col).cast(column_types[c]))
        print(f"Bronze records: {bronze.count()}")

        # 6. CHECK NULL BUSINESS KEYS
        null_condition = None
        for key in business_keys:
            condition = col(key).is_null()
            null_condition = (
                condition
                if null_condition is None
                else null_condition | condition
            )
        if bronze.filter(null_condition).limit(1).count() > 0:
            raise ValueError(f"Null business key found in {source_table}")
        print("Business key NULL validation passed.")

        # 7. IDENTIFY AND LOG DUPLICATE BUSINESS KEYS
        duplicate_keys = (
            bronze
            .group_by(*[C(key) for key in business_keys])
            .count()
            .filter(C("COUNT") > 1)
        )
        
        duplicate_rows = bronze.join(
            duplicate_keys.select(
                *[C(key) for key in business_keys],
                C("COUNT").alias("DUPLICATE_COUNT")
            ),
            on=[key.strip('"').upper() for key in business_keys],
            how="inner"
        )
        
        duplicate_count = duplicate_log(
            duplicate_rows,
            run_id,
            source_table,
            business_keys,
            data_columns
        )
        
        print(f"Duplicate records logged: {duplicate_count}")
        
        bronze = bronze.join(
            duplicate_keys.select(
                *[C(key) for key in business_keys]
            ),
            on=[key.strip('"').upper() for key in business_keys],
            how="left_anti"
        )


        # 8. VALIDATE BRONZE BUSINESS KEYS
        validate_business_keys(bronze,business_keys,source_table)

        # 9. GENERATE BRONZE HASH KEY
        bronze = generate_hash_key(bronze,hash_columns,hash_name)

        # 10. READ CURRENT SILVER RECORDS
        silver = (session.table(target_table).filter(col(latest_col) == latest_value).select(*data_columns))
        print(f"Current Silver records: {silver.count()}")
        validate_business_keys(silver,business_keys,target_table)

        # Generate hash for current Silver records
        silver = generate_hash_key(silver,hash_columns,hash_name)

        
        # 11. COMPARE BRONZE AND SILVER
        b = bronze.alias("B")
        s = silver.with_column("_FOUND", lit(1)).alias("S")
        # Build join condition using DataFrame alias column references
        join_condition = None
        for key in business_keys:
            key_name = key.strip('"').upper()
            condition = (
                b[key_name] == s[key_name]
            )
        
            join_condition = (
                condition
                if join_condition is None
                else join_condition & condition
            )
        
        comparison = b.join(
            s,
            join_condition,
            "left"
        )
        
        # New records: business key does not exist in current Silver
        is_new = s["_FOUND"].is_null()
        
        # Changed records: key exists but hash is different
        is_changed = (
            s["_FOUND"].is_not_null()
            & (
                b[hash_name.upper()]
                != s[hash_name.upper()]
            )
        )
        
        # Extract new records
        new_records = (
            comparison
            .filter(is_new)
            .select(
                *[
                    b[c.strip('"').upper()].alias(c.strip('"').upper())
                    for c in bronze.columns
                ]
            )
        )
        
        # Extract changed records
        changed_records = (
            comparison
            .filter(is_changed)
            .select(
                *[
                    b[c.strip('"').upper()].alias(c.strip('"').upper())
                    for c in bronze.columns
                ]
            )
        )
        
        new_record_count = new_records.count()
        changed_record_count = changed_records.count()
        
        print(f"New records: {new_record_count}")
        print(f"Changed records: {changed_record_count}")


        # 12. PREPARE NEW AND CHANGED VERSIONS
        new_versions = (new_records.union_all(changed_records).with_column(inserted_col,F.current_timestamp())
            .with_column(updated_col,lit(dummy_date).cast(column_types[updated_col]))
            .with_column(latest_col,lit(latest_value).cast(column_types[latest_col]))
            .select(*column_names)
        )

        # 13. PREPARE EXPIRED RECORDS
        expire_records = changed_records.select(*business_keys)

        # Add NULL values for remaining columns
        for c in column_names:
            if c not in business_keys:
                expire_records = expire_records.with_column(c, lit(None).cast(column_types[c]))
        expire_records = expire_records.select(*column_names).with_column("_SCD_ACTION", lit("EXPIRE"))

        # Prepare INSERT action records
        insert_records = new_versions.with_column("_SCD_ACTION",lit("INSERT"))

        # Combine EXPIRE and INSERT records
        merge_source = expire_records.union_all(insert_records)

        # 14. EXECUTE SNOWFLAKE MERGE FOR SCD TYPE 2
        if new_record_count > 0 or changed_record_count > 0:
            # Unique temporary table name
            safe_table_name = re.sub(r"[^A-Za-z0-9_]","_",table_name).upper()
            temp_table = (f"CUSTOMER_BILLING_DB.SILVER."f"TMP_SCD_{safe_table_name}_{run_id}")

            # Store merge source in a temporary Snowflake table
            merge_source.write.save_as_table(temp_table,mode="overwrite",table_type="temporary")

            # Match current records by business keys
            merge_keys = " AND ".join(
                [
                    f"T.{key} = S.{key}"
                    for key in business_keys
                ]
            )

            merge_condition = (
                f"{merge_keys} "
                f"AND T.{latest_col} = {int(latest_value)} "
                f"AND S._SCD_ACTION = 'EXPIRE'"
            )

            # Build INSERT column and value lists
            insert_columns = ", ".join(column_names)
            insert_values = ", ".join(
                [f"S.{c}" for c in column_names]
            )

            merge_sql = f"""
                MERGE INTO {target_table} T
                USING {temp_table} S
                    ON {merge_condition}
                    
                WHEN MATCHED
                    AND S._SCD_ACTION = 'EXPIRE'
                THEN UPDATE SET
                    T.{latest_col} = 0,
                    T.{updated_col} = CURRENT_TIMESTAMP()

                WHEN NOT MATCHED
                    AND S._SCD_ACTION = 'INSERT'
                THEN INSERT (
                    {insert_columns}
                )
                VALUES (
                    {insert_values}
                )
            """

            print("Executing Snowflake SCD Type 2 MERGE...")
            merge_result = session.sql(merge_sql).collect()

            # Insert count = new records + changed versions
            inserted_count = (new_record_count + changed_record_count)

            # Updated count = expired old versions
            updated_count = changed_record_count
            print("Snowflake MERGE completed.")

            # Drop temporary merge source
            session.sql(f"DROP TABLE IF EXISTS {temp_table}").collect()
        else:
            print("No new or changed records found.")

        # 15. SUCCESS LOGGING
        process_log("Success",table_name=table_name,layer="silver",total_inserted_count=inserted_count,total_updated_count=updated_count)
        print(f"Completed {target_table}: "f"inserted={inserted_count}, "f"updated={updated_count}")

    # 16. ERROR HANDLING
    except Exception as e:
        error_text = str(e)
        print(f"Failed {table_name}: {error_text}")
        try:
            process_log("Failed",table_name=table_name,layer="silver",total_inserted_count=inserted_count,total_updated_count=updated_count,error_message=error_text)
        except Exception as log_error:
            print(f"Could not write failure log: {log_error}")
        raise